# 01 - Setup: Carga de Dados no SQL Server

Este notebook carrega os arquivos CSV da pasta `data/` para o banco de dados **Ecommerce** no SQL Server 2025.

**Pré-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- SQL Server acessível na porta `1433`
- Driver ODBC 18 instalado (`sudo apt install unixodbc-dev`)

## 1. Configuração e Conexão

In [1]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

Servidor: localhost:1433
Database: Ecommerce


In [2]:
# Conexão ao SQL Server (master) para criar o database
conn_master = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao SQL Server (master) com sucesso!')

Conectado ao SQL Server (master) com sucesso!


## 2. Criar Database Ecommerce

In [3]:
# Criar o database se não existir
cursor_master.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_DATABASE}')
    BEGIN
        CREATE DATABASE [{DB_DATABASE}]
    END
""")
print(f'Database [{DB_DATABASE}] criado/verificado com sucesso!')
cursor_master.close()
conn_master.close()

Database [Ecommerce] criado/verificado com sucesso!


In [4]:
# Conectar ao database SeguroDB
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso!')

Conectado ao [Ecommerce] com sucesso!


## 3. Criar Tabelas

In [5]:
ddl_statements = [
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'categorias')
    CREATE TABLE categorias (
        id_categoria INT PRIMARY KEY,
        nome_categoria VARCHAR(100),
        descricao VARCHAR(300)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'produtos')
    CREATE TABLE produtos (
        id_produto INT PRIMARY KEY,
        nome VARCHAR(200),
        id_categoria INT,
        preco DECIMAL(10,2),
        estoque INT
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'clientes')
    CREATE TABLE clientes (
        id_cliente INT PRIMARY KEY,
        nome VARCHAR(200),
        estado CHAR(2),
        status_conta VARCHAR(20)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'vendas')
    CREATE TABLE vendas (
        id_venda INT PRIMARY KEY,
        id_cliente INT,
        data_venda DATE,
        valor_total DECIMAL(10,2)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'itens_venda')
    CREATE TABLE itens_venda (
        id_item INT PRIMARY KEY,
        id_venda INT,
        id_produto INT,
        quantidade INT,
        preco_unitario DECIMAL(10,2)
    )
    """
]

for ddl in ddl_statements:
    cursor.execute(ddl)
    
print('Todas as tabelas foram criadas com sucesso!')

Todas as tabelas foram criadas com sucesso!


## 4. Carregar Dados dos CSVs

In [6]:
# Ordem de carga (respeitar dependências)
tabelas = [
    'categorias', 'produtos', 'clientes', 'vendas', 'itens_venda'
]

diretorio_atual = os.getcwd()
if diretorio_atual.endswith('notebook'):
    raiz_projeto = os.path.dirname(diretorio_atual)
else:
    raiz_projeto = diretorio_atual

data_dir = os.path.join(raiz_projeto, 'data')

for tabela in tabelas:
    csv_path = os.path.join(data_dir, f'{tabela}.csv')
    
    # Verificar se a tabela já tem dados
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    
    if count > 0:
        print(f'{tabela}: já contém {count} registros, pulando...')
        continue
    
    # Ler CSV
    df = pd.read_csv(csv_path)
    
    # Limpar espaços em colunas string
    for col in df.select_dtypes(include=['str', 'object']).columns:
        df[col] = df[col].str.strip()
    
    # Inserir dados via executemany
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f'INSERT INTO {tabela} ({cols}) VALUES ({placeholders})'
    
    # Converter NaN para None
    data = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]
    
    # Inserir em lotes de 1000
    batch_size = 2000
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        cursor.executemany(insert_sql, batch)
    
    print(f'{tabela}: {len(data)} registros inseridos')

print('\nCarga de dados concluída!')

categorias: já contém 5 registros, pulando...
produtos: já contém 10 registros, pulando...
clientes: já contém 10 registros, pulando...
vendas: já contém 10 registros, pulando...
itens_venda: já contém 13 registros, pulando...

Carga de dados concluída!


## 5. Validação

In [7]:
# Verificar contagem de registros em cada tabela
print(f'{"Tabela":<20} {"Registros":>10}')
print('-' * 32)

for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<20} {count:>10}')

print('\nValidação concluída!')

Tabela                Registros
--------------------------------
categorias                    5
produtos                     10
clientes                     10
vendas                       10
itens_venda                  13

Validação concluída!


In [8]:
# Amostra de dados de algumas tabelas
tabelas_delta = ['categorias', 'produtos', 'clientes', 'vendas', 'itens_venda']

for tabela in tabelas_delta:
    print(f'\n--- {tabela.upper()} (primeiros 5 registros) ---')
    df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
    print(df_sample.to_string(index=False))


--- CATEGORIAS (primeiros 5 registros) ---
 id_categoria nome_categoria                        descricao
            1    Eletrônicos Smartphones, notebooks e gadgets
            2      Vestuário    Roupas masculinas e femininas
            3         Livros    Ficção, técnicos e literatura
            4           Casa      Móveis e itens de decoração
            5       Esportes Artigos esportivos e suplementos

--- PRODUTOS (primeiros 5 registros) ---
 id_produto               nome  id_categoria  preco  estoque
          1     Smartphone XYZ             1 1500.0       50
          2       Notebook Pro             1 3500.0       30
          3    Camiseta Básica             2   50.0      100
          4        Calça Jeans             2  120.0       80
          5 O Senhor dos Anéis             3   60.0       40

--- CLIENTES (primeiros 5 registros) ---
 id_cliente           nome estado status_conta
          1     João Silva     SP        Ativo
          2 Maria Oliveira     RJ       

/tmp/ipykernel_19426/2161777575.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
/tmp/ipykernel_19426/2161777575.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
/tmp/ipykernel_19426/2161777575.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
/tmp/ipykernel_19426/2161777575.py:6: UserWarning: pandas only supports SQLAlchemy conne

In [9]:
# Encerrar conexão
cursor.close()
conn.close()
print('Conexão encerrada.')

Conexão encerrada.
